# Auto-labelling and Training a YOLOv8 Detector with Autodistill

Builds a custom object detector without manual annotation: frames are sampled from
video, auto-labelled by GroundedSAM from a text prompt (an *ontology*), and the
resulting dataset trains a YOLOv8 model. The trained weights are then run on new
clips.

Based on Roboflow's autodistill walkthrough, adapted for a final-year project on
locating a target of interest in surveillance footage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip /content/dataset.zip -d /content/

In [ ]:
!pip install -q \
autodistill \
autodistill-grounded-sam \
autodistill-yolov8 \
roboflow \
supervision==0.24.0

In [ ]:
# Colab occasionally reports a non-UTF-8 preferred encoding, which breaks pip here
import locale
def getpreferredencoding(do_setlocale=True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

!pip install albumentations==1.4

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
!mkdir {HOME}/images
!mkdir {HOME}/images2
!mkdir {HOME}/videos

## Sample frames from the source videos

Each clip is decoded and every *n*-th frame is written out as a training image.
The stride is tuned per clip to balance dataset size against redundancy.

In [ ]:
import supervision as sv
import os

HOME = "/content"
IMAGE_DIR_PATH = f"{HOME}/images"

# (video path, frame stride) pairs
SOURCES = [
    ("/content/drive/MyDrive/video2/video2.mov", 8),
    ("/content/drive/MyDrive/video2/video3.mov", 8),
    ("/content/drive/MyDrive/video2/video4.mov", 7),
]

for video_path, frame_stride in SOURCES:
    video_name = os.path.basename(video_path)
    image_name_pattern = video_name + "-{:05d}.png"
    with sv.ImageSink(target_dir_path=IMAGE_DIR_PATH, image_name_pattern=image_name_pattern) as sink:
        for image in sv.get_video_frames_generator(source_path=str(video_path), stride=frame_stride):
            sink.save_image(image=image)

In [ ]:
import supervision as sv

image_paths = sv.list_files_with_extensions(
    directory=IMAGE_DIR_PATH,
    extensions=["png", "jpg"])

print('image count:', len(image_paths))

In [ ]:
IMAGE_DIR_PATH = f"{HOME}/images"
SAMPLE_SIZE = 16
SAMPLE_GRID_SIZE = (4, 4)
SAMPLE_PLOT_SIZE = (16, 16)

In [ ]:
import cv2
import supervision as sv

titles = [
    image_path.stem
    for image_path
    in image_paths[:SAMPLE_SIZE]]
images = [
    cv2.imread(str(image_path))
    for image_path
    in image_paths[:SAMPLE_SIZE]]

# Preview a grid of sampled frames
sv.plot_images_grid(images=images, titles=titles, grid_size=SAMPLE_GRID_SIZE, size=SAMPLE_PLOT_SIZE)

## Define the ontology

The ontology maps a natural-language description of the target to the class label
that GroundedSAM should assign. Two variants were used: one for a specific vehicle,
and one for two people distinguished by clothing.

In [ ]:
from autodistill.detection import CaptionOntology

# Target vehicle described by make, model and colour (plate number omitted)
ontology = CaptionOntology({
    "Lexus UX250h in aurora titanium color": "Goal"
})

In [ ]:
from autodistill.detection import CaptionOntology

# Alternative ontology: two people distinguished by clothing
ontology = CaptionOntology({
    "A woman wearing green long sleeves": "P1",
    "A man wearing a black jacket, black trousers, a backpack and glasses": "P2"
})

In [ ]:
DATASET_DIR_PATH = f"{HOME}/dataset"

In [ ]:
import torch
torch.use_deterministic_algorithms(False)

In [ ]:
import os
os.makedirs(DATASET_DIR_PATH, exist_ok=True)

from autodistill_grounded_sam import GroundedSAM

# GroundedSAM auto-labels every frame according to the ontology
base_model = GroundedSAM(ontology=ontology)

dataset = base_model.label(
    input_folder=IMAGE_DIR_PATH,
    extension=".png",
    output_folder=DATASET_DIR_PATH)

In [ ]:
ANNOTATIONS_DIRECTORY_PATH = f"{HOME}/dataset/train/labels"
IMAGES_DIRECTORY_PATH = f"{HOME}/dataset/train/images"
DATA_YAML_PATH = f"{HOME}/dataset/data.yaml"

In [ ]:
import supervision as sv

dataset = sv.DetectionDataset.from_yolo(
    images_directory_path=IMAGES_DIRECTORY_PATH,
    annotations_directory_path=ANNOTATIONS_DIRECTORY_PATH,
    data_yaml_path=DATA_YAML_PATH)

len(dataset)

In [ ]:
import supervision as sv
from pathlib import Path

mask_annotator = sv.MaskAnnotator()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

images = []
image_names = []
for i, (image_path, image, annotation) in enumerate(dataset):
    if i == SAMPLE_SIZE:
        break
    annotated_image = image.copy()
    annotated_image = mask_annotator.annotate(
        scene=annotated_image, detections=annotation)
    annotated_image = box_annotator.annotate(
        scene=annotated_image, detections=annotation)
    annotated_image = label_annotator.annotate(
        scene=annotated_image, detections=annotation)

    image_names.append(Path(image_path).name)
    images.append(annotated_image)

sv.plot_images_grid(
    images=images,
    titles=image_names,
    grid_size=SAMPLE_GRID_SIZE,
    size=SAMPLE_PLOT_SIZE)

In [ ]:
!zip -r dataset.zip dataset

## Train YOLOv8 on the auto-labelled dataset

An initial run trains from the pretrained `yolov8n.pt`; a second run can resume
from the previous `best.pt` for additional epochs.

In [ ]:
%cd {HOME}

from autodistill_yolov8 import YOLOv8

target_model = YOLOv8("yolov8n.pt")
target_model.train(DATA_YAML_PATH, epochs=90)

In [ ]:
# Optional: continue training from the previous best checkpoint
%cd {HOME}

from autodistill_yolov8 import YOLOv8

target_model = YOLOv8("/content/best.pt")
target_model.train(DATA_YAML_PATH, epochs=80)

## Inspect training results

In [ ]:
!ls {HOME}/runs/detect/train/

In [ ]:
%cd {HOME}
from IPython.display import Image
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
%cd {HOME}
from IPython.display import Image
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
%cd {HOME}
from IPython.display import Image
Image(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=800)

## Run the trained model on new clips

Swap `INPUT_VIDEO_PATH` and the confidence threshold per clip. A higher `conf`
keeps only the most certain detections of the target.

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

INPUT_VIDEO_PATH = "/content/drive/MyDrive/video2/video1.mov"
TRAINED_MODEL_PATH = f"{HOME}/runs/detect/train/weights/best.pt"

!yolo predict model={TRAINED_MODEL_PATH} source={INPUT_VIDEO_PATH} conf=0.90 iou=0.1

In [ ]:
# Example with a different clip and a lower confidence threshold
INPUT_VIDEO_PATH = "/content/drive/MyDrive/video2_1/video2-5.MOV"
TRAINED_MODEL_PATH = "/content/runs/detect/train2/weights/best.pt"

!yolo predict model={TRAINED_MODEL_PATH} source={INPUT_VIDEO_PATH} conf=0.45 iou=0